<a href="https://colab.research.google.com/github/hasimcode/hasimcode/blob/main/Nuclear_Power_Plants_Grid_Connection_Yearly_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib

# Configure matplotlib for video writing
matplotlib.rcParams['animation.writer'] = 'ffmpeg'
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# 1. Load and clean data
df = pd.read_csv('reactors_data.csv')
df = df.dropna(subset=['Year', 'Connections'])
df['Year'] = pd.to_numeric(df['Year']).astype(int)
df['Connections'] = pd.to_numeric(df['Connections'])
df = df.sort_values('Year').reset_index(drop=True)
df['Cumulative'] = df['Connections'].cumsum()

# Define key historical event milestones
historical_events = {
    1954: "1954: First Grid Connection\n(Obninsk, USSR)",
    1986: "1986: Chernobyl Incident\n(Global Safety Reviews)",
    2011: "2011: Fukushima Incident\"
}

# Precompute when cumulative reactor thresholds are first crossed (100, 200, 300, etc.)
threshold_milestones = {}
for threshold in range(100, int(df['Cumulative'].max()) + 1, 100):
    crossing_row = df[df['Cumulative'] >= threshold].first_valid_index()
    if crossing_row is not None:
        year = int(df.loc[crossing_row, 'Year'])
        # Store the exact cumulative value at that year to position annotation correctly
        actual_val = df.loc[crossing_row, 'Cumulative']
        threshold_milestones[year] = (actual_val, f"{threshold} Reactors Crossed\n(Year {year})")

# 2. Setup Figure and Axes with an elegant light theme
fig, ax = plt.subplots(figsize=(12, 7), facecolor='#fcfcfc')
ax.set_facecolor('#fcfcfc')

# Initializing plot elements
line, = ax.plot([], [], lw=3, color='#0f4c81', zorder=3, label='Cumulative Connections')
scatter = ax.scatter([], [], color='#ff6f61', s=80, zorder=4)  # Current progress marker
fill = ax.fill_between([], [], color='#0f4c81', alpha=0.15, zorder=2)

# Styling the axes
ax.set_xlim(df['Year'].min() - 1, df['Year'].max() + 1)
ax.set_ylim(0, df['Cumulative'].max() * 1.1)

# Premium Typography & Labels
ax.set_title('Global Nuclear Reactor Grid Connections (1954 - 2026)', fontsize=16, fontweight='bold', pad=20, color='#2c3e50')
ax.set_xlabel('Year', fontsize=12, labelpad=10, color='#2c3e50')
ax.set_ylabel('Cumulative Grid Connections', fontsize=12, labelpad=10, color='#2c3e50')
ax.tick_params(colors='#7f8c8d', labelsize=10)
ax.grid(True, linestyle='--', alpha=0.5, color='#bdc3c7')

# Remove top and right spines
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.spines['left'].set_color('#bdc3c7')
ax.spines['bottom'].set_color('#bdc3c7')

# Text display for current status
stats_text = ax.text(0.05, 0.9, '', transform=ax.transAxes, fontsize=12,
                     fontweight='bold', color='#2c3e50', bbox=dict(facecolor='white', alpha=0.8, edgecolor='#bdc3c7', boxstyle='round,pad=0.5'))

# 3. Animation Logic
def init():
    line.set_data([], [])
    scatter.set_offsets(np.empty((0, 2)))
    stats_text.set_text('')
    return line, scatter, stats_text

def update(frame):
    # Remove older temporary annotations to avoid overlapping
    for txt in list(ax.texts):
        if txt != stats_text:
            txt.remove()

    x = df['Year'].iloc[:frame]
    y = df['Cumulative'].iloc[:frame]

    # Update primary line
    line.set_data(x, y)

    # Update fill area
    global fill
    fill.remove()
    fill = ax.fill_between(x, y, color='#0f4c81', alpha=0.15, zorder=2)

    # Update active point
    if len(x) > 0:
        current_year = x.iloc[-1]
        current_val = y.iloc[-1]
        scatter.set_offsets(np.array([[current_year, current_val]]))
        stats_text.set_text(f"Year: {int(current_year)}\nTotal: {int(current_val)}")

        # Dynamically place persistent historical milestones
        for yr, label in historical_events.items():
            if current_year >= yr:
                yr_idx = df[df['Year'] == yr].index[0]
                yr_val = df['Cumulative'].iloc[yr_idx]
                ax.annotate(label,
                            xy=(yr, yr_val),
                            xytext=(yr - 12 if yr != 1954 else yr + 2, yr_val + 50 if yr == 1986 else yr_val + 30 if yr == 1954 else yr_val - 70),
                            arrowprops=dict(arrowstyle="->", color='#ff6f61', lw=1.5),
                            fontsize=9, fontweight='bold', color='#2c3e50',
                            bbox=dict(facecolor='#ffffff', alpha=0.9, edgecolor='#1abc9c', boxstyle='round,pad=0.3'))

        # Dynamically place persistent threshold crossing milestones (100, 200, 300, etc.)
        for yr, (val, label) in threshold_milestones.items():
            if current_year >= yr:
                ax.annotate(label,
                            xy=(yr, val),
                            xytext=(yr - 14, val + 50 if val < 400 else val - 70),
                            arrowprops=dict(arrowstyle="->", color='#3498db', lw=1.2, linestyle='--'),
                            fontsize=8, fontweight='normal', color='#2c3e50',
                            bbox=dict(facecolor='#f0f4f8', alpha=0.9, edgecolor='#3498db', boxstyle='round,pad=0.3'))
    else:
        scatter.set_offsets(np.empty((0, 2)))
        stats_text.set_text('')

    return line, scatter, stats_text

# 4. Save enhanced animation
anim = FuncAnimation(fig, update, frames=len(df)+1, init_func=init, blit=False, interval=200)
anim.save('cumulative_reactors.mp4', writer='ffmpeg', fps=10, dpi=300)
plt.close(fig)

In [ ]:
import io
import base64
from IPython.display import HTML

# Read and encode the newly styled video
video_path = 'cumulative_reactors.mp4'
with open(video_path, 'r+b') as f:
    video_bytes = f.read()

encoded = base64.b64encode(video_bytes).decode('ascii')

# Display the styled HTML5 video element
HTML(f'''
<div style="text-align: center;">
    <video width="720" height="540" controls style="border: 1px solid #bdc3c7; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1);">
        <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
</div>
''')

In [ ]:
from google.colab import files

# Trigger the browser download for the generated video
files.download('cumulative_reactors.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>